# nanoPeriGPT — Incremental Experiments

Peridynamic attention + DEER layer-parallelism for transformers.

Each cell runs one experiment. Compare val_loss between cells to track progress.

In [ ]:
# Setup: upload project files or clone from repo
import os
import torch

# If running from uploaded zip:
# !unzip -q perigpt.zip -d perigpt
# os.chdir('perigpt')

# If cloning from git:
# !git clone <your-repo-url> perigpt
# os.chdir('perigpt')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
# Prepare data
!python data/shakespeare_char/prepare.py

In [ ]:
# Phase 1: Vanilla nanoGPT Baseline
!python train.py config/baseline.py

In [ ]:
# Phase 2: Bond-Based Peridynamic Attention (horizon=32)
!python train.py config/baseline.py \
    --out_dir=out-peri-h32 \
    --wandb_run_name=peri_h32 \
    --experiment_name=peri_h32 \
    --attention_type=peridynamic \
    --horizon=32

In [ ]:
# Phase 3: Horizon Sweep
for h in [16, 64, 128]:
    print(f'\n{"="*60}')
    print(f'Horizon = {h}')
    print(f'{"="*60}')
    !python train.py config/baseline.py \
        --out_dir=out-peri-h{h} \
        --wandb_run_name=peri_h{h} \
        --experiment_name=peri_h{h} \
        --attention_type=peridynamic \
        --horizon={h}

In [ ]:
# Check results so far
import pandas as pd
df = pd.read_csv('results.tsv', sep='\t')
print(df.sort_values('val_loss').to_string(index=False))

In [ ]:
# Phase 5: Block Attention Residual (standard attn + block_attn depth)
!python train.py config/baseline.py \
    --out_dir=out-blockres \
    --wandb_run_name=block_attn_res \
    --experiment_name=block_attn_res \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 6: Combined — Peri Attention + Block Attn Residual
# Use best horizon from Phase 3
BEST_HORIZON = 32  # <-- update based on Phase 3 results
!python train.py config/baseline.py \
    --out_dir=out-peri-blockres \
    --wandb_run_name=peri_blockres \
    --experiment_name=peri_blockres \
    --attention_type=peridynamic \
    --horizon={BEST_HORIZON} \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Phase 11: Hybrid Attention (Peri local + sparse global)
!python train.py config/baseline.py \
    --out_dir=out-hybrid \
    --wandb_run_name=hybrid_a8 \
    --experiment_name=hybrid_a8 \
    --attention_type=hybrid \
    --horizon={BEST_HORIZON} \
    --n_global_anchors=8

In [ ]:
# Final results comparison
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('results.tsv', sep='\t')
df = df.sort_values('val_loss')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if s == 'KEEP' else 'red' for s in df['status']]
ax.barh(df['experiment'], df['val_loss'], color=colors)
ax.set_xlabel('Validation Loss')
ax.set_title('nanoPeriGPT Experiments — shakespeare_char')
ax.invert_yaxis()
for i, (v, t) in enumerate(zip(df['val_loss'], df['train_loss'])):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('progress.png', dpi=150)
plt.show()
print(df.to_string(index=False))

In [ ]:
# Sample from best model
BEST_DIR = 'out-baseline'  # <-- update to best experiment's out_dir
!python sample.py --out_dir={BEST_DIR} --num_samples=3 --max_new_tokens=500

# Mixed-Domain Experiments

The critical test: does peridynamic damage learn to break bonds at domain boundaries?

Dataset: Shakespeare + Python code + JSON + Federalist Papers interleaved in random chunks (64-384 chars). Domain boundaries fall within the 256-char context window.

In [ ]:
# Prepare mixed-domain data
!python data/mixed_domain/prepare.py

In [ ]:
# Mixed-domain: Baseline (standard attention)
!python train.py config/mixed_baseline.py

In [ ]:
# Mixed-domain: Peridynamic attention (bond-based) — horizon sweep
for h in [32, 64, 128]:
    print(f'\n{"="*60}')
    print(f'Mixed-domain Peridynamic h={h}')
    print(f'{"="*60}')
    !python train.py config/mixed_baseline.py \
        --out_dir=out-mixed-peri-h{h} \
        --experiment_name=mixed_peri_h{h} \
        --attention_type=peridynamic \
        --horizon={h}

In [ ]:
# Mixed-domain: Block attention residual
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-blockres \
    --experiment_name=mixed_block_attn_res \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Mixed-domain: Peri + Block Attn Res combined
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-peri-blockres \
    --experiment_name=mixed_peri_blockres \
    --attention_type=peridynamic \
    --horizon=64 \
    --residual_type=block_attn \
    --depth_block_size=3

In [ ]:
# Mixed-domain: Hybrid attention
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-hybrid \
    --experiment_name=mixed_hybrid_a8 \
    --attention_type=hybrid \
    --horizon=64 \
    --n_global_anchors=8

In [ ]:
# Mixed-domain: State-based peridynamic attention (THE BIG ONE)
for h in [64, 128]:
    !python train.py config/mixed_baseline.py \
        --out_dir=out-mixed-state-h{h} \
        --experiment_name=mixed_state_peri_h{h} \
        --attention_type=state_peridynamic \
        --horizon={h}

In [ ]:
# Compare ALL results — shakespeare vs mixed-domain
import pandas as pd

df = pd.read_csv('results.tsv', sep='\t')
df = df.sort_values('val_loss')

print("=" * 80)
print("ALL EXPERIMENTS")
print("=" * 80)
print(df.to_string(index=False))

# Separate shakespeare vs mixed results
print("\n" + "=" * 80)
print("SHAKESPEARE (homogeneous)")
print("=" * 80)
shk = df[~df['experiment'].str.startswith('mixed')]
print(shk.to_string(index=False))

print("\n" + "=" * 80)
print("MIXED-DOMAIN (heterogeneous)")
print("=" * 80)
mix = df[df['experiment'].str.startswith('mixed')]
print(mix.to_string(index=False))

# The key question: does peri attention rank differently on mixed vs shakespeare?
print("\n" + "=" * 80)
print("KEY COMPARISON: Peri vs Baseline ranking change")
print("=" * 80)
for dataset_name, sub in [("Shakespeare", shk), ("Mixed", mix)]:
    if len(sub) < 2:
        continue
    baseline = sub[sub['experiment'].str.contains('baseline')]['val_loss'].values
    peri = sub[sub['experiment'].str.contains('peri_h')]['val_loss'].values
    if len(baseline) > 0 and len(peri) > 0:
        best_peri = min(peri)
        b = baseline[0]
        diff = best_peri - b
        direction = "WORSE" if diff > 0 else "BETTER"
        print(f"  {dataset_name}: baseline={b:.4f}, best_peri={best_peri:.4f}, "
              f"delta={diff:+.4f} ({direction})")

In [ ]:
# Mixed-domain: Increased bond_dim (bond_dim_ratio=2 means bd=32 instead of 16)
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-peri-bd2 \
    --experiment_name=mixed_peri_bd2 \
    --attention_type=peridynamic \
    --horizon=128 \
    --bond_dim_ratio=2

In [ ]:
# Mixed-domain: Block attn-res WITH damage gating (fixes cross-domain contamination)
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-blockdmg \
    --experiment_name=mixed_blockres_damage \
    --residual_type=block_attn \
    --depth_block_size=3 \
    --block_damage=True

In [ ]:
# Mixed-domain: Full stack — state-based peri + damage-gated block_attn_res
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-full \
    --experiment_name=mixed_full_stack \
    --attention_type=state_peridynamic \
    --horizon=128 \
    --residual_type=block_attn \
    --depth_block_size=3 \
    --block_damage=True

In [ ]:
# === TRITON SPEED BENCHMARK ===
# Compare all kernel variants: flash, PyTorch peri, Triton generic, Triton Blackwell
import time
import torch
from model import GPTConfig, GPT

def bench(attn_type, label, horizon=128, n_runs=50, warmup=10):
    cfg = GPTConfig(block_size=256, vocab_size=65, n_layer=6, n_head=6, n_embd=384,
                    dropout=0.0, bias=False, deer_enabled=False, energy_lambda=0.0,
                    attention_type=attn_type, horizon=horizon, bond_dim_ratio=2)
    model = GPT(cfg).cuda().eval()
    x = torch.randint(0, 65, (64, 256), device='cuda')
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            model(x)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / n_runs * 1000
    print(f'{label:40s} {ms:8.2f} ms/step  params={model.get_num_params()/1e6:.2f}M')
    del model
    torch.cuda.empty_cache()
    return ms

t_std = bench('standard', 'Standard (flash) attention')
t_swa = bench('sliding_window', 'Sliding window h=128')
t_peri = bench('peridynamic', 'Peridynamic (PyTorch)')
t_triton = bench('peridynamic_triton', 'Peridynamic (Triton generic)')
t_blackwell = bench('peridynamic_blackwell', 'Peridynamic (Triton Blackwell)')

print(f'\n{"="*60}')
print(f'Standard (flash):     {t_std:8.2f} ms  (1.00x)')
print(f'Sliding window:       {t_swa:8.2f} ms  ({t_swa/t_std:.2f}x)')
print(f'Peri PyTorch:         {t_peri:8.2f} ms  ({t_peri/t_std:.2f}x)')
print(f'Peri Triton generic:  {t_triton:8.2f} ms  ({t_triton/t_std:.2f}x)')
print(f'Peri Triton Blackwell:{t_blackwell:8.2f} ms  ({t_blackwell/t_std:.2f}x)')
print(f'\nBlackwell vs PyTorch: {t_peri/t_blackwell:.2f}x speedup')
print(f'Blackwell vs generic: {t_triton/t_blackwell:.2f}x speedup')

# Literature Comparison Baselines

Three baselines from published work, run on the same data for fair comparison:
1. **Sliding Window Attention** (Longformer/Mistral) — same horizon as peri, but standard Q*K scoring, no damage
2. **Kimi Attention Residuals** (arXiv:2603.15031) — same block attention over depth, but no damage gating
3. **DenseFormer DWA** (NeurIPS 2024) — learned scalar weights over depth instead of attention

# Damage Analysis

The key scientific question: does damage correlate with domain boundaries?

In [ ]:
# Run damage analysis on the best peri model
# Update out_dir to match your best mixed-domain peri checkpoint
!python analyze_damage.py --out_dir=out-mixed-peri-h128 --n_samples=100

In [ ]:
# 1. Sliding Window Attention (Longformer/Mistral style)
#    Same bounded horizon as peridynamic, but standard Q*K scoring, NO damage
#    If peri beats this, the improvement is from strain/damage, not just the window
for h in [32, 128]:
    !python train.py config/mixed_baseline.py \
        --out_dir=out-mixed-swa-h{h} \
        --experiment_name=mixed_swa_h{h} \
        --attention_type=sliding_window \
        --horizon={h}

In [ ]:
# 2. Kimi Attention Residuals (arXiv:2603.15031, March 2026)
#    Same block attention over depth as ours, but NO damage gating
#    If our block_attn beats this, damage gating is the contribution
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-kimi \
    --experiment_name=mixed_kimi_attn_res \
    --residual_type=kimi_attn_res \
    --depth_block_size=3

In [ ]:
# 3. DenseFormer DWA (Pagliardini et al., NeurIPS 2024)
#    Learned scalar weights over depth (simpler than attention)
!python train.py config/mixed_baseline.py \
    --out_dir=out-mixed-denseformer \
    --experiment_name=mixed_denseformer \
    --residual_type=denseformer

In [ ]:
# === ALSO RUN LITERATURE BASELINES ON SHAKESPEARE ===
# (so we have apples-to-apples comparison on both datasets)

# Sliding window on shakespeare
for h in [32, 128]:
    !python train.py config/baseline.py \
        --out_dir=out-swa-h{h} \
        --experiment_name=swa_h{h} \
        --attention_type=sliding_window \
        --horizon={h}

# Kimi AttnRes on shakespeare
!python train.py config/baseline.py \
    --out_dir=out-kimi \
    --experiment_name=kimi_attn_res \
    --residual_type=kimi_attn_res \
    --depth_block_size=3

# DenseFormer on shakespeare
!python train.py config/baseline.py \
    --out_dir=out-denseformer \
    --experiment_name=denseformer \
    --residual_type=denseformer

In [ ]:
# === DOWNLOAD ALL RESULTS AND CHECKPOINTS ===
import os
import zipfile
from google.colab import files

zip_name = 'perigpt_results.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    # results.tsv
    if os.path.exists('results.tsv'):
        zf.write('results.tsv')
        print(f"  + results.tsv ({os.path.getsize('results.tsv')} bytes)")

    # Each out-* directory: checkpoints + analysis files
    for d in sorted(os.listdir('.')):
        if d.startswith('out-') and os.path.isdir(d):
            for f in os.listdir(d):
                fpath = os.path.join(d, f)
                if f.endswith('.pt') or f.endswith('.pkl'):
                    zf.write(fpath)
                    size_mb = os.path.getsize(fpath) / 1e6
                    print(f"  + {fpath} ({size_mb:.1f} MB)")

print(f"\nTotal: {os.path.getsize(zip_name) / 1e6:.1f} MB")
files.download(zip_name)

In [ ]:
# === FULL COMPARISON TABLE ===
import pandas as pd

df = pd.read_csv('results.tsv', sep='\t')

# Tag each experiment with its dataset
df['dataset'] = df['experiment'].apply(lambda x: 'mixed' if x.startswith('mixed') else 'shakespeare')

# Tag method type for grouping
def get_method(name):
    if 'swa' in name: return 'Sliding Window (Longformer/Mistral)'
    if 'kimi' in name: return 'Kimi AttnRes (2026)'
    if 'denseformer' in name: return 'DenseFormer DWA (NeurIPS 2024)'
    if 'state_peri' in name: return 'State-Based Peri (ours)'
    if 'peri' in name and 'block' in name: return 'Peri + BlockRes (ours)'
    if 'peri' in name and 'bd' in name: return 'Peri bond_dim (ours)'
    if 'blockres_damage' in name: return 'BlockRes + Damage (ours)'
    if 'block_attn' in name: return 'Block AttnRes (ours)'
    if 'full_stack' in name: return 'Full Stack (ours)'
    if 'hybrid' in name: return 'Hybrid Peri+Global (ours)'
    if 'peri' in name: return 'Bond-Based Peri (ours)'
    if 'baseline' in name: return 'Standard Attention'
    return 'Other'

df['method'] = df['experiment'].apply(get_method)

for dataset in ['shakespeare', 'mixed']:
    sub = df[df['dataset'] == dataset].sort_values('val_loss')
    print(f"\n{'='*80}")
    print(f"  {dataset.upper()} — sorted by val_loss")
    print(f"{'='*80}")
    print(sub[['experiment', 'val_loss', 'train_loss', 'method']].to_string(index=False))

# Key ablation: peri vs sliding window (isolates damage contribution)
print(f"\n{'='*80}")
print("  KEY ABLATION: Peridynamic vs Sliding Window (same horizon)")
print("  Difference = contribution of strain-based scoring + damage")
print(f"{'='*80}")
for dataset in ['shakespeare', 'mixed']:
    sub = df[df['dataset'] == dataset]
    for h in [32, 128]:
        peri = sub[sub['experiment'].str.contains(f'peri_h{h}') & ~sub['experiment'].str.contains('block|state|bd')]
        swa = sub[sub['experiment'].str.contains(f'swa_h{h}')]
        if len(peri) > 0 and len(swa) > 0:
            p_val = peri['val_loss'].values[0]
            s_val = swa['val_loss'].values[0]
            diff = p_val - s_val
            winner = "PERI" if diff < 0 else "SWA"
            print(f"  {dataset} h={h}: peri={p_val:.4f} swa={s_val:.4f} delta={diff:+.4f} ({winner})")

# Key ablation: our block_attn_res vs Kimi (isolates damage gating)
print(f"\n{'='*80}")
print("  KEY ABLATION: Our Block AttnRes vs Kimi AttnRes (isolates damage gating)")
print(f"{'='*80}")
for dataset in ['shakespeare', 'mixed']:
    sub = df[df['dataset'] == dataset]
    ours = sub[sub['experiment'].str.contains('block_attn_res') & ~sub['experiment'].str.contains('kimi|mixed_peri|damage')]
    kimi = sub[sub['experiment'].str.contains('kimi')]
    if len(ours) > 0 and len(kimi) > 0:
        o_val = ours['val_loss'].values[0]
        k_val = kimi['val_loss'].values[0]
        diff = o_val - k_val
        winner = "OURS" if diff < 0 else "KIMI"
        print(f"  {dataset}: ours={o_val:.4f} kimi={k_val:.4f} delta={diff:+.4f} ({winner})")